In [ ]:
# Signal Processing of Conductivity Fluctuations

This notebook processes experimental conductivity fluctuation data from thin metal films.

Material groups:
- BI — bismuth films
- CR — chromium films
- FE — iron films

Reference group:
- XXX — modal reference film used to identify composition-independent noise

Pipeline:
1. Read binary `.Q00` files.
2. Split signals into overlapping samples.
3. Remove linear trends.
4. Apply a Blackman window.
5. Compute FFT-based power spectra.
6. Extract 1/f noise spectrum parameters and average spectral energy.
7. Save the resulting dataset.

In [ ]:
from pathlib import Path
import struct

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.fft import rfft
from scipy.signal.windows import blackman

In [ ]:
DATA_DIR = Path("../data/q00")
OUTPUT_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results/figures")

SAMPLE_LENGTH = 128
OVERLAP_MODE = 2
SAMPLING_RATE_HZ = 20000

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FILE_NAMES = [
    "BI202C01",
    "BI304C01",
    "CR202C01",
    "CR304C01",
    "FE104C01",
    "FE202C01",
    "XXX01C01",
    "XXX01C03",
    "XXX01C05",
]

REFERENCE_LABEL = "XXX"
MATERIAL_ORDER = ["BI", "CR", "FE"]

In [ ]:
## Read binary `.Q00` files

Each `.Q00` file contains metadata and raw ADC codes.  
The codes are converted to physical units and split into fixed-length samples.

In [ ]:
def read_q00_file(file_path: Path):
    with file_path.open("rb") as f:
        km = struct.unpack("<f", f.read(4))[0]
        ku = struct.unpack("<f", f.read(4))[0]

        while True:
            header = f.read(32)

            if len(header) < 32:
                break

            jd1, jd2, up1, up2, kc, n_values = struct.unpack(
                "<ddffii",
                header,
            )

            raw_codes = f.read(n_values * 2)

            if len(raw_codes) < n_values * 2:
                break

            codes = np.frombuffer(raw_codes, dtype="<i2")

            yield {
                "km": km,
                "ku": ku,
                "jd1": jd1,
                "jd2": jd2,
                "up1": up1,
                "up2": up2,
                "kc": kc,
                "codes": codes,
            }

In [ ]:
def split_into_samples(codes, sample_length=128, overlap_mode=2):
    if overlap_mode == 0:
        step = sample_length
    elif overlap_mode == 1:
        step = int(sample_length * 3 / 4)
    elif overlap_mode == 2:
        step = int(sample_length / 2)
    elif overlap_mode == 3:
        step = int(sample_length / 4)
    else:
        raise ValueError("overlap_mode must be 0, 1, 2, or 3.")

    samples = []

    for start in range(0, len(codes) - sample_length + 1, step):
        samples.append(codes[start:start + sample_length])

    return samples

In [ ]:
## Spectral feature extraction

For each detrended sample, the notebook computes:

- `gamma`: slope of the log-log power spectrum, associated with 1/f noise;
- `energy`: mean spectral power.

In [ ]:
def compute_spectrum_features(sample, sampling_rate=SAMPLING_RATE_HZ):
    n = len(sample)

    if n < 2:
        return np.nan, np.nan

    window = blackman(n)
    windowed_signal = sample * window

    spectrum = rfft(windowed_signal)
    power = np.abs(spectrum) ** 2

    frequencies = np.fft.rfftfreq(n, d=1 / sampling_rate)

    mask = frequencies > 0
    frequencies = frequencies[mask]
    power = power[mask]

    positive = power > 0
    frequencies = frequencies[positive]
    power = power[positive]

    if len(frequencies) < 2:
        return np.nan, np.nan

    gamma, _ = np.polyfit(
        np.log(frequencies),
        np.log(power),
        1,
    )

    energy = float(np.mean(power))

    return gamma, energy

In [ ]:
def process_q00_file(
    file_path,
    sample_length=128,
    overlap_mode=2,
    sampling_rate=SAMPLING_RATE_HZ,
):
    records = list(read_q00_file(file_path))

    if not records:
        raise ValueError(f"No valid records found in {file_path.name}")

    features = []

    for record in records:
        scale = 2.0 / (
            record["kc"]
            * record["ku"]
            * record["km"]
            * (record["up1"] + record["up2"])
        )

        samples = split_into_samples(
            codes=record["codes"],
            sample_length=sample_length,
            overlap_mode=overlap_mode,
        )

        time_index = np.arange(sample_length)

        for sample in samples:
            sample = np.asarray(sample, dtype=float)

            linear_coeffs = np.polyfit(time_index, sample, 1)
            trend = np.polyval(linear_coeffs, time_index)

            detrended_sample = (sample - trend) * scale

            gamma, energy = compute_spectrum_features(
                detrended_sample,
                sampling_rate=sampling_rate,
            )

            features.append(
                {
                    "gamma": gamma,
                    "energy": energy,
                    "jd1": record["jd1"],
                    "jd2": record["jd2"],
                }
            )

    return pd.DataFrame(features)

In [ ]:
## Process all files

The material label is extracted from the file name.  
XXX files are marked as the modal reference film.

In [ ]:
def extract_material(file_stem):
    name = file_stem.upper()

    if name.startswith("BI"):
        return "BI"
    if name.startswith("CR"):
        return "CR"
    if name.startswith("FE"):
        return "FE"
    if name.startswith("XXX"):
        return "XXX"

    return "UNKNOWN"


all_frames = []

for file_stem in FILE_NAMES:
    file_path = DATA_DIR / f"{file_stem}.Q00"

    if not file_path.exists():
        print(f"Skipping missing file: {file_path}")
        continue

    df = process_q00_file(
        file_path=file_path,
        sample_length=SAMPLE_LENGTH,
        overlap_mode=OVERLAP_MODE,
    )

    df["file"] = file_stem
    df["material"] = extract_material(file_stem)

    output_path = OUTPUT_DIR / f"{file_stem}_features.csv"
    df.to_csv(output_path, index=False)

    all_frames.append(df)

    print(f"{file_stem}: {len(df)} samples processed")

dataset = pd.concat(all_frames, ignore_index=True)
dataset = dataset.dropna(subset=["gamma", "energy"])

dataset.to_csv(OUTPUT_DIR / "conductivity_features.csv", index=False)

dataset.head()

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=dataset,
    x="gamma",
    y="energy",
    hue="material",
    hue_order=MATERIAL_ORDER + [REFERENCE_LABEL],
    alpha=0.6,
)

plt.yscale("log")
plt.xlabel("Gamma")
plt.ylabel("Mean spectral energy")
plt.title("Conductivity fluctuation features by material and reference film")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "feature_distribution.png", dpi=300)
plt.show()

In [ ]:
## Output

The notebook creates:

- Individual feature files in `data/processed/`
- A combined dataset: `data/processed/conductivity_features.csv`
- A feature distribution plot in `results/figures/`

The resulting dataset contains:
- `gamma`: 1/f noise spectrum slope
- `energy`: mean spectral power
- `material`: film group label
- `XXX`: modal reference film used for composition-independent noise analysis